# D5-信令客流洞察-数据流水线与性能

In [ ]:
# ── 第一格：环境准备 ──
%load_ext autoreload
%autoreload 2
import sqlite3
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from signal_insight import config, geo, quality, viz
from signal_insight.viz import MAIN, MAIN_LIGHT, MAIN_DARK, MAIN_MID, GRAY, GRAY_DARK, RED, AMBER, INK, MUTED, FAINT
viz.apply_theme()
con = sqlite3.connect(config.DB_PATH)
DAY = config.ANALYSIS_DAY
NB = "D05_流水线与性能"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}；制图：{config.AUTHOR}"
pd.set_option("display.max_columns", 20)


## 为什么要做流水线？

甲方说：这份数据以后每天都会来一份，处理要能每天自动跑。回头看前四天：从 ODS 到 dwd_signal 的清洗散在 D02 十几个格子里，dim_cell 在 D03、dim_hotspot 在 D04，要打开三本 notebook 按顺序手动运行。

notebook 干不了这件事：定时任务没法“打开浏览器点运行”；格子可以乱序执行，跑完的 notebook 不等于从上到下跑一遍的结果（D04 修的 D3_02 就是例子）。

## 数据清洗封装: dwd.py

In [ ]:
from signal_insight import dwd
raw = pd.read_sql("SELECT * FROM ods_signal", con)
dwd_new, steps = dwd.clean_signal(raw, DAY)
dwd_d02 = pd.read_sql("SELECT * FROM dwd_signal WHERE dt = ?",
                    con,
                    params=(DAY,),
                    parse_dates=['ts']  
                    )
pd.DataFrame(steps)

In [ ]:
# 逐列对账：哪些列和 D02 的表不一样？
diff = [c for c in dwd_new.columns if not (dwd_new[c].astype(str) == dwd_d02[c].astype(str)).all()]
print("不一致的列：", diff or "无")
print("库里 ts 带毫秒的行：", int((dwd_d02["ts"].dt.microsecond > 0).sum()),
      "；函数版带毫秒的行：", int((dwd_new["ts"].dt.microsecond > 0).sum()))

## 读写数据收进db.py: 按日期区分结果不变

In [ ]:
# 按日期分区写两遍：结果不变（幂等）；顺便用正确的时间格式把 D02 丢掉的毫秒补回来
from signal_insight import db
for _ in range(2):
    db.write_partition(con, dwd_new, "dwd_signal", DAY)
n = pd.read_sql("SELECT COUNT(*) AS n FROM dwd_signal WHERE dt = ?",
                con,
                params=(DAY, ))["n"][0]
back = pd.read_sql("SELECT ts FROM dwd_signal WHERE dt = ?",
                   con,
                   params=(DAY, ),
                   parse_dates=["ts"])
print(
    f"写了两遍，{DAY} 仍是 {n:,} 行；读回来的时间空值 {back['ts'].isna().sum()} 个；\
带毫秒的行 {(back['ts'].dt.microsecond > 0).sum():,}"
)

## 维度表: 收紧dims.py

流水线要一起生成dim_cell 和 dim_hotspot
- dims_cell: 按照坐标定站点，编号，落区县
- dim_hotspot: 连接用户前5的站点按用户数加权

In [ ]:
from signal_insight import dims

districts = geo.rings_from_points(
    pd.read_sql("SELECT * FROM ods_district_boundary", con),"district")

aois = geo.rings_from_points(
    pd.read_sql("SELECT * FROM ods_aoi_v2", con), "name"
)
dim_new = dims.build_dim_cell(
    pd.read_sql("SELECT * FROM ods_cell", con), districts, aois)
result = pd.testing.assert_frame_equal(dim_new,
        pd.read_sql("SELECT * FROM dim_cell", con), check_dtype=False
                            )
hot_new = dims.build_dim_hotspot(dwd_new, dim_new)
hot_d04 = pd.read_sql("SELECT * FROM dim_hotspot", con)
hot_d04  # 正常来说hot_do4和hot_new是一致的

## 数据流水线
- dwd, db, dims表都有了，还差一步：把流程串起来，每一步留下记录，并在终端中一条命令跑完

In [ ]:
from signal_insight import pipeline

log = pipeline.run_day() # 执行流水线流程
chk = pd.read_sql(
    "SELECT (SELECT COUNT(*) FROM dwd_signal WHERE dt = '2018-10-03') AS dwd, "
    "(SELECT COUNT(*) FROM dim_cell) AS dim_cell, (SELECT name FROM dim_hotspot) AS hotspot, "
    "(SELECT COUNT(DISTINCT 批次) FROM etl_log) AS runs", con)
etl_log = log[["步骤", "输入行数", "输出行数", "耗时(秒)"]]

In [ ]:
etl_log

In [ ]:
# 对账：流水线跑两次，结果与前几天一致
assert chk.loc[0, "dwd"] == 515_931 and chk.loc[0, "dim_cell"] == 49_948 and chk.loc[0, "hotspot"] == "西塔"
assert chk.loc[0, "runs"] >= 2
print(f"✅ 对账通过：dwd_signal 515,931 行，dim_cell 49,948 行，热点为西塔；etl_log 里有 {chk.loc[0, 'runs']} 次运行")

## 关于数据读取相关优化
Polars 和 Pandas 是 Python 中两个主流的 DataFrame 库。**Pandas 是成熟、生态丰富的行业标准，适合通用数据分析；而 Polars 是后起之秀，专为高性能和大型数据集设计，在速度、内存效率和并行处理上优势显著。**

### 📊 核心对比速览

| 特性 | Pandas | Polars |
| :--- | :--- | :--- |
| **底层实现** | Python (C/Cython 内部) | Rust (通过 PyO3 提供 Python 绑定) |
| **内存模型** | 传统上基于 NumPy (行式存储)，2.0 起可选 Arrow | 基于 Apache Arrow (列式存储) |
| **执行模式** | 仅支持即时求值 (Eager) | 支持即时求值 (Eager) 和惰性求值 (Lazy) |
| **并行能力** | 默认单线程，需借助 Dask 等库实现并行 | 自动多线程，利用所有 CPU 核心 |
| **索引系统** | 有行索引，支持 `.loc[]`、`.iloc[]` 等复杂索引 | 无行索引，仅支持整数位置索引 |
| **类型系统** | 相对宽松，允许隐式类型转换 | 严格类型，需要显式转换 |
| **流式处理** | 不支持，数据必须放入内存 | 支持流式处理，可处理大于内存的数据集 |

### 🚀 性能与效率：为什么 Polars 更快？

Polars 的性能优势源于其现代化的架构设计：

- **自动并行与惰性求值**：Polars 自动将操作并行到所有 CPU 核心。其**惰性求值**模式会先构建查询计划，通过**谓词下推、投影下推**等优化技术，在读取数据前就裁剪掉不必要的工作，从而大幅提升效率。
- **列式存储与 Arrow 格式**：数据以 Apache Arrow 列式格式存储，对 CPU 缓存更友好，并支持 SIMD 指令，使得列操作极快。
- **实际基准测试**：
    - **CSV 读取**：在一个 5000 万行的数据集测试中，Polars 耗时约 52 秒，而 Pandas 耗时 **6分12秒**，差距约 **7 倍**。在 100 万行数据的测试中，Polars 读取速度是 Pandas 的 **8.2 倍**。
    - **分组聚合**：有基准测试显示，对 1000 万行数据执行一次 `groupby`，Polars 仅需 **0.45 秒**，而 Pandas 需要 **12.5 秒**，速度相差近 **28 倍**。
- **内存效率**：Polars 的内存需求显著更低。有分析指出，Pandas 执行操作时大约需要数据集大小 **5-10 倍**的 RAM，而 Polars 仅需 **2-4 倍**。在实际的 Pipeline 测试中，Polars 的内存峰值比 Pandas **减少了约 60%**。

### 💻 语法与 API 差异

从 Pandas 迁移到 Polars，需要适应一些关键的心智模型变化：

- **无索引系统**：Polars 的 DataFrame 没有行索引。你无法再使用 `df.loc[]` 或 `df.iloc[]`，所有基于标签或位置的选取都通过 `filter()` 和列表达式完成。
- **表达式 API**：这是 Polars 的核心。不再使用 `df["col"]`，而是使用 `pl.col("col")` 来构建可组合、可优化的表达式。
- **方法链**：Polars 鼓励使用方法链（如 `.filter().with_columns().group_by()`）来构建数据处理管道，而不是像 Pandas 那样频繁进行原地赋值。
- **严格类型**：Polars 要求显式的类型转换。例如，Pandas 在整数列中插入 `None` 会自动转为浮点型，而 Polars 会保留为整数类型并标记为 `null`。

**操作对照示例**：

| 操作 | Pandas | Polars |
| :--- | :--- | :--- |
| **选择列** | `df[["a", "b"]]` | `df.select("a", "b")` |
| **过滤** | `df[df["age"] > 25]` | `df.filter(pl.col("age") > 25)` |
| **新增列** | `df["new"] = df["old"] * 2` | `df.with_columns(new=pl.col("old") * 2)` |
| **条件逻辑** | `np.where(cond, a, b)` | `pl.when(cond).then(a).otherwise(b)` |

### 🌐 生态系统与兼容性

- **Pandas 的优势**：拥有**无可比拟的生态系统**。它与 `scikit-learn`、`Matplotlib`、`Seaborn` 等 PyData 核心库深度集成，几乎所有数据科学项目都依赖它。它支持最广泛的 I/O 格式，包括 Excel、HDF5、SAS 等。
- **Polars 的追赶**：Polars 基于 Arrow，因此能与任何支持 Arrow 的库**零拷贝**地共享数据，例如 DuckDB、PyArrow。其生态系统正在快速增长，许多库已原生支持 Polars。但需要注意的是，部分机器学习库可能仍需要你将 Polars DataFrame 转换为 Pandas 格式后再输入。Polars 对 Excel 的支持也需要通过转换实现，不如 Pandas 直接。

### 🎯 如何选择：场景与建议

**优先选择 Pandas 的场景：**
- **数据量较小**（例如少于 100 万行），且能完全放入内存。
- **需要与机器学习生态深度集成**，如 `scikit-learn`、`statsmodels`。
- **团队已熟悉 Pandas**，且培训成本高于性能收益。
- **频繁处理 Excel 文件**或使用**冷门 I/O 格式**（如 HDF5, SAS）。
- **进行快速、临时的探索性分析**，Pandas 的熟悉度和即时反馈更便捷。

**优先选择 Polars 的场景：**
- **数据量较大**（超过 100 万行），且性能瓶颈开始显现。
- **构建生产级数据流水线**，需要高性能和低资源消耗。
- **内存是主要约束**，Polars 的内存效率优势明显。
- **需要并行处理但不想引入 Dask 等复杂框架**，Polars 自动并行。
- **数据主要以 Parquet 格式存储**，Polars 的谓词下推能带来巨大效率提升。
- **处理大于内存的数据集**，Polars 的流式处理能力是原生支持。

### 🔄 迁移策略

不必一次性重写所有代码。官方推荐采用**分段迁移**的策略：
1.  **定位瓶颈**：找出 Pipeline 中性能最慢、内存消耗最大的环节。
2.  **局部替换**：只将该环节的数据处理逻辑用 Polars 重写，在环节的入口和出口处使用 `.to_pandas()` 和 `pl.from_pandas()` 进行格式转换。
3.  **度量验证**：对比迁移前后的运行时间和内存峰值，确保获得收益且结果正确。

这种方式可以让你以最小的风险，逐步享受 Polars 带来的性能红利。

### 💎 总结

**Polars 并非 Pandas 的简单替代品，而是一个针对特定问题域的现代化解决方案**。对于追求极致性能、处理大规模数据、构建高效数据管道的场景，Polars 是更优的选择。而对于生态集成、交互式探索和中小规模数据分析，Pandas 依然是那个最成熟、最稳妥的基石。

两者并非互斥，你完全可以在同一个项目中根据任务特点混合使用它们。

## 性能优化：数据库选择
DuckDB 和 SQLite 虽然都是轻量级的嵌入式数据库，但它们的设计目标截然不同，**核心区别在于：SQLite 是为事务处理（OLTP）而生的“应用文件格式”，而 DuckDB 是为分析处理（OLAP）而生的“本地分析引擎”**。

### 📊 核心对比速览

| 特性 | SQLite | DuckDB |
| :--- | :--- | :--- |
| **核心定位** | OLTP (事务型) | OLAP (分析型) |
| **存储模型** | 行式存储 (Row-store) | 列式存储 (Columnar) |
| **执行引擎** | 逐行处理，基于B-tree | 向量化执行，多线程并行 |
| **最佳场景** | 点查、插入、更新、删除 | 聚合、分组、大表关联 |
| **并发能力** | 多读单写 (WAL模式可多读并行) | 单进程内多线程读，并发写受限 |
| **文件大小** | 极小 (约 750KB) | 较小 (约 20MB) |
| **数据类型** | 动态类型，较宽松 | 严格类型，支持数组/结构体等 |
| **生态位** | 应用内本地存储的事实标准 | 数据科学工作流中的“迷你数仓” |

### 🏗️ 架构分歧：行与列的对决

两者性能差异的根源在于存储架构。

*   **SQLite 采用行式存储**：整行数据连续存放，这使得它在**按主键查询单条记录**时极为高效，写入也很快。但执行 `SUM(amount)` 这类聚合时，它需要读取整行数据，效率较低。
*   **DuckDB 采用列式存储**：每列数据独立存放。当执行 `SUM(amount)` 时，它**只需扫描 `amount` 这一列**，数据量更小，压缩率更高，对 CPU 缓存也更友好。代价是单行更新或插入的开销较大。

### 🚀 性能表现：场景决定胜负

#### OLTP 场景：SQLite 胜出

在涉及大量单行写入、更新的场景下，SQLite 性能更优。基准测试显示，在某些单行操作中，**SQLite 的速度大约是 DuckDB 的 2 倍**。例如，在一次测试中，SQLite 的批量插入和单次插入提交耗时分别为 47.5µs 和 14.1µs，而 DuckDB 则分别为 1.04ms 和 277.5µs。

#### OLAP 场景：DuckDB 大幅领先

在分析型查询上，DuckDB 的优势是压倒性的。有报告指出，DuckDB 的分析查询速度比 SQLite **快 3 到 5 倍**，且数据量越大优势越明显。在 ClickBench 基准测试中，某些分析查询上 DuckDB 的性能甚至**快了多达 1200 倍**。

#### 存储效率：DuckDB 优势明显

DuckDB 的列式存储和压缩算法使其文件体积更小。在一个实际案例中，存储 100 万行数据，SQLite 文件大小为 **101.6MB**，而 DuckDB 仅需 **23.1MB**，**体积减少了约 77%**。

### 🎯 如何选择：场景与建议

**优先选择 SQLite 的场景：**
*   **应用内本地存储**：移动 App、桌面软件、浏览器等的配置或用户数据存储。
*   **嵌入式与物联网**：资源受限的边缘设备，需要零管理、极小体积的数据库。
*   **事务性工作负载**：需要频繁进行插入、更新、删除操作，且要求 ACID 事务保证。
*   **简单的键值查找**：基于主键或索引的快速点查。

**优先选择 DuckDB 的场景：**
*   **本地数据分析**：在 Jupyter Notebook 中直接对 CSV/Parquet 文件执行复杂的 SQL 聚合查询，无需预先导入。
*   **数据科学工作流**：与 Pandas、Polars、Arrow 等无缝集成，作为数据管道中的高速分析引擎。
*   **构建数据管道**：处理日志、事件等中等规模数据（5M-50M 行），进行 ETL 或即席分析。
*   **内存敏感的场景**：需要处理接近内存极限的数据集，DuckDB 的高压缩率和高效扫描能降低内存压力。

### 🔄 混合架构：各取所长

两者并非互斥。一个强大的模式是**混合使用**：用 **SQLite 处理应用的事务性写入**，同时用 **DuckDB 连接同一个 SQLite 文件进行快速分析**。DuckDB 的 `ATTACH` 命令可以直接挂载 SQLite 数据库文件，实现无缝的 OLAP 分析。

### 💎 总结

**SQLite 是应用可靠的事务性“记录员”，而 DuckDB 是数据科学家高效的“分析引擎”**。选择哪一个，完全取决于你的工作负载是偏向高频的“记录”操作，还是偏向复杂的“洞察”分析。在数据驱动的应用中，理解它们的差异并让它们各司其职，往往是最佳的工程实践。

## 把前面的对帐改成测试

会有两种测试：
- contract: 检查契约：每个模块对外长什么样子，允许做到什么，不允许做什么？测试不通过的时候就改自己的代码，不改测试-这是企业验收的标准
- unit: 这个是你自己的，D02, D05的对帐数字搬过来，再加几条边界情况，比如空输入，不修改输入,测试目录的结构可以对照src: 测dwd.py的就叫test_dwd.py

## 从稽核表到数据质量报告

在这一周里面我们做的每一项检查都进入了dq_check_result，把它汇成一张图，作为数据质量报告的第一张图：一共查了几项，哪几项有问题，问题集中在什么维度

In [ ]:
# D02 登记稽核时，有两条编号写成了下划线（DQ_02、DQ_03）。排序时 "_" 排在 "-" 后面，这两条会跑到 DQ-16 之后。
# 统一成短横线。D02 notebook 里那两处 dq_record 也要同步改成 DQ-02、DQ-03，否则重跑 D02 又会写回来。
con.execute("UPDATE dq_check_result SET check_id = REPLACE(check_id, '_', '-') WHERE check_id IN ('DQ_02', 'DQ_03')")
con.commit()
ids = pd.read_sql("SELECT check_id FROM dq_check_result", con)["check_id"]
assert ids.is_unique and ids.str.fullmatch(r"DQ-\d{2}").all(), ids[~ids.str.fullmatch(r"DQ-\d{2}")].tolist()
print(f"✅ 稽核表 {len(ids)} 条，编号唯一、格式统一（DQ-01 … DQ-{ids.str[3:].astype(int).max():02d}）")


In [ ]:
dq = pd.read_sql("SELECT * FROM dq_check_result", con).sort_values("check_id")
summary = (dq.pivot_table(index="维度", columns="状态", values="check_id", aggfunc="count", fill_value=0)
             .reindex(columns=["不通过", "警告", "通过"], fill_value=0))
summary["问题数"] = summary["不通过"] + summary["警告"]
summary = summary.sort_values(["问题数", "不通过"])
summary.to_sql("ads_dq_summary", con, if_exists="replace")


In [ ]:
dq.to_csv("./数据质量检查表.csv",index=None,encoding='utf-8')

In [ ]:
# ── 出图 D5_03：堆叠条形。三种状态互斥且穷尽，可以堆；每段写状态词，不只靠颜色 ──
# 标题里的数字：tot = 三种状态各自的总项数（t_ 按列求和）。
t_ = dq.pivot_table(index="维度",
            columns="状态",
            values="check_id", 
            aggfunc="count", fill_value=0)
t_ = t_.reindex(columns=["不通过", "警告", "通过"],
            fill_value=0).sort_values(["不通过", "警告"], ascending=False)
tot = t_.sum()
STATUS = {"不通过": RED, "警告": AMBER, "通过": GRAY}         # 每段都写状态词：不只靠颜色，色弱读者也能读
fig, ax = plt.subplots(figsize=(10, 5.6), layout="constrained")
left = np.zeros(len(t_))
for kk in ["不通过", "警告", "通过"]:
    ax.barh(range(len(t_)), t_[kk], left=left,
        color=STATUS[kk], height=0.6, edgecolor="white", lw=1.2)
    for i, v in enumerate(t_[kk]):
        if v:
            ax.text(left[i] + v / 2, i, f"{kk} {v}" if v > 1 else f"{v}",
                    ha="center", va="center", fontsize=9.5,
                    color="white" if kk != "通过" else INK, fontweight="bold")
    left += t_[kk].to_numpy()
for i, v in enumerate(left):
    ax.text(v + 0.08, i, f"共 {int(v)} 项", va="center", fontsize=9, color=MUTED)
ax.set_yticks(range(len(t_)), t_.index, fontsize=10, color=INK); ax.invert_yaxis()
ax.set_xlim(0, left.max() + 1); ax.set_xticks([]); ax.grid(False)
title = f"{int(tot.sum())} 项检查只有 {tot['通过']} 项直接通过：{tot['不通过']} 项不通过、{tot['警告']} 项警告"
viz.finish(fig, title,
           "各质量维度的检查结果（项数），按不通过、警告的项数排序；红 = 不通过，黄 = 警告，灰 = 通过；口径：dq_check_result，D1–D5 全部检查",
           SOURCE, "D5_03_数据质量检查汇总.png")
plt.show()
